In [3]:
# from langchain_community.chat_models import ChatOllama
import json
import pandas as pd
from docx import Document

In [6]:
bsc_path = "../Data/raw/bsc.xlsx"  # update path

bsc_df = pd.read_excel(bsc_path)

bsc_df.head()

,Major Objectives,Major – KPI,Target
0,Ensure Sustainable Profitability,Economic Profit,Achieve Economic Profit Plan
1,Ensure Sustainable Profitability,Economic Profit,Graduating branches from loss to profit making
2,Enhance Resource Mobilization,Incremental Deposit,Achieve Incremental Deposit plan
3,Enhance Resource Mobilization,Incremental Deposit,Mobilize Deposit from Retail Customers
4,Enhance Resource Mobilization,Incremental Deposit,Mobilize Deposit from Wholesale Customers


In [ ]:
bsc_df.columns = [col.strip().lower().replace(" ", "_") for col in bsc_df.columns]
bsc_df.columns

Index(['major_objectives', 'major_–_kpi', 'target'], dtype='object')

In [8]:
bsc_df.head()

,major_objectives,major_–_kpi,target
0,Ensure Sustainable Profitability,Economic Profit,Achieve Economic Profit Plan
1,Ensure Sustainable Profitability,Economic Profit,Graduating branches from loss to profit making
2,Enhance Resource Mobilization,Incremental Deposit,Achieve Incremental Deposit plan
3,Enhance Resource Mobilization,Incremental Deposit,Mobilize Deposit from Retail Customers
4,Enhance Resource Mobilization,Incremental Deposit,Mobilize Deposit from Wholesale Customers


In [ ]:
bsc_filtered = bsc_df[["strategic_objective", "kpi"]].dropna()

bsc_filtered = bsc_filtered.drop_duplicates()

bsc_filtered.head()

In [10]:
def format_bsc_context(df):
    context = ""
    
    for _, row in df.iterrows():
        context += f"""
Strategic Objective: {row['major_objectives']}
KPI: {row['major_–_kpi']}
target: {row['target']}
------------------------
"""
    return context

bsc_context = format_bsc_context(bsc_df)

print(bsc_context[:1000])


Strategic Objective: Ensure Sustainable  Profitability
KPI: Economic Profit
target: Achieve  Economic Profit Plan
------------------------

Strategic Objective: Ensure Sustainable  Profitability
KPI: Economic Profit
target: Graduating branches from loss to profit making
------------------------

Strategic Objective: Enhance  Resource Mobilization
KPI: Incremental Deposit
target: Achieve  Incremental Deposit plan
------------------------

Strategic Objective: Enhance  Resource Mobilization
KPI: Incremental Deposit
target: Mobilize Deposit from Retail Customers
------------------------

Strategic Objective: Enhance  Resource Mobilization
KPI: Incremental Deposit
target: Mobilize Deposit from Wholesale Customers
------------------------

Strategic Objective: Enhance  Resource Mobilization
KPI: Incremental Deposit
target: Mobilize Deposit from CBE NOOR Customers
------------------------

Strategic Objective: Enhance  Resource Mobilization
KPI: Incremental Deposit
target: Mobilize Deposit 

In [11]:
bsc_context

"\nStrategic Objective: Ensure Sustainable  Profitability\nKPI: Economic Profit\ntarget: Achieve  Economic Profit Plan\n------------------------\n\nStrategic Objective: Ensure Sustainable  Profitability\nKPI: Economic Profit\ntarget: Graduating branches from loss to profit making\n------------------------\n\nStrategic Objective: Enhance  Resource Mobilization\nKPI: Incremental Deposit\ntarget: Achieve  Incremental Deposit plan\n------------------------\n\nStrategic Objective: Enhance  Resource Mobilization\nKPI: Incremental Deposit\ntarget: Mobilize Deposit from Retail Customers\n------------------------\n\nStrategic Objective: Enhance  Resource Mobilization\nKPI: Incremental Deposit\ntarget: Mobilize Deposit from Wholesale Customers\n------------------------\n\nStrategic Objective: Enhance  Resource Mobilization\nKPI: Incremental Deposit\ntarget: Mobilize Deposit from CBE NOOR Customers\n------------------------\n\nStrategic Objective: Enhance  Resource Mobilization\nKPI: Incremental 

In [43]:
jd_path = "../Data/raw/JD1.docx"

doc = Document(jd_path)

len(doc.tables)

1

In [46]:
table = doc.tables[0]  # 👈 only first role

In [47]:
def table_to_text(table):
    text = ""
    for row in table.rows:
        row_text = " | ".join([cell.text.strip() for cell in row.cells if cell.text.strip()])
        text += row_text + "\n"
    return text.lower()

jd_text = table_to_text(table)

print(jd_text[:1000])

job details /profile/: | job details /profile/: | job details /profile/:
job title: branch banking officer | organizational relationships: | organizational relationships:
job code:(to be filled by hr) | reports directly to: customer service manager-sales | reports directly to: customer service manager-sales
division: retail & branch banking | supervises: none | supervises: none
department: district | type of employment: permanent | type of employment: permanent
job grade: 9
unit:  branch | job category: professional: p | job category: professional: p
job objective: to sale bank products and services, expands branch digital banking users, attained customers request and provides efficient and high-quality service to customers as per the bank’s policy, procedure, guideline and other pertinent regulation. | job objective: to sale bank products and services, expands branch digital banking users, attained customers request and provides efficient and high-quality service to customers as per t

In [48]:
def clean_text(text):
    text = re.sub(r'\s+', ' ', text)
    return text

jd_text = clean_text(jd_text)

In [49]:
def extract_field(text, field_name):
    pattern = rf"{field_name}\s*:\s*([^|\n]+)"
    match = re.search(pattern, text, re.IGNORECASE)
    
    if match:
        return match.group(1).strip()
    
    return None

In [50]:
jd_data = {
    "job_title": extract_field(jd_text, "job title"),
    "division": extract_field(jd_text, "division"),
    "department": extract_field(jd_text, "department"),
    "job_grade": extract_field(jd_text, "job grade"),
    "job_category": extract_field(jd_text, "job category"),
    "job_objective": extract_field(jd_text, "job objective"),
}

jd_data

{'job_title': 'branch banking officer',
 'division': 'retail & branch banking',
 'department': 'district',
 'job_grade': '9 unit: branch',
 'job_category': 'professional: p',
 'job_objective': 'to sale bank products and services, expands branch digital banking users, attained customers request and provides efficient and high-quality service to customers as per the bank’s policy, procedure, guideline and other pertinent regulation.'}

In [51]:
def clean_value(val):
    if val:
        return val.replace("|", "").strip()
    return None

jd_cleaned = {k: clean_value(v) for k, v in jd_data.items()}

jd_cleaned

{'job_title': 'branch banking officer',
 'division': 'retail & branch banking',
 'department': 'district',
 'job_grade': '9 unit: branch',
 'job_category': 'professional: p',
 'job_objective': 'to sale bank products and services, expands branch digital banking users, attained customers request and provides efficient and high-quality service to customers as per the bank’s policy, procedure, guideline and other pertinent regulation.'}

In [52]:
def format_jd_context(jd):
    return f"""
Job Title: {jd.get('job_title')}
Division: {jd.get('division')}
Department: {jd.get('department')}
Job Grade: {jd.get('job_grade')}
Job Category: {jd.get('job_category')}

Job Objective:
{jd.get('job_objective')}
"""

jd_context = format_jd_context(jd_cleaned)

print(jd_context)


Job Title: branch banking officer
Division: retail & branch banking
Department: district
Job Grade: 9 unit: branch
Job Category: professional: p

Job Objective:
to sale bank products and services, expands branch digital banking users, attained customers request and provides efficient and high-quality service to customers as per the bank’s policy, procedure, guideline and other pertinent regulation.



In [55]:
full_context = f"""
=== Job Description ===
{jd_context}

=== Balanced Scorecard ===
{bsc_context}


"""

print(full_context[:1500])


=== Job Description ===

Job Title: branch banking officer
Division: retail & branch banking
Department: district
Job Grade: 9 unit: branch
Job Category: professional: p

Job Objective:
to sale bank products and services, expands branch digital banking users, attained customers request and provides efficient and high-quality service to customers as per the bank’s policy, procedure, guideline and other pertinent regulation.


=== Balanced Scorecard ===

Strategic Objective: Ensure Sustainable  Profitability
KPI: Economic Profit
target: Achieve  Economic Profit Plan
------------------------

Strategic Objective: Ensure Sustainable  Profitability
KPI: Economic Profit
target: Graduating branches from loss to profit making
------------------------

Strategic Objective: Enhance  Resource Mobilization
KPI: Incremental Deposit
target: Achieve  Incremental Deposit plan
------------------------

Strategic Objective: Enhance  Resource Mobilization
KPI: Incremental Deposit
target: Mobilize Deposi

In [69]:
def generate_prompt(context, num_objectives):
    prompt = f"""
You are an AI assistant. Based on the following job description and BSC, generate exactly {num_objectives} SMART objectives.

{full_context}

Please return the objectives in a numbered list.
"""
    return prompt

In [61]:
from gpt4all import GPT4All

In [ ]:
promt = generate_prompt(full_context, 3)

"\nYou are an AI assistant. Based on the following job description and BSC, generate exactly 3 SMART objectives.\n\n\n=== Job Description ===\n\nJob Title: branch banking officer\nDivision: retail & branch banking\nDepartment: district\nJob Grade: 9 unit: branch\nJob Category: professional: p\n\nJob Objective:\nto sale bank products and services, expands branch digital banking users, attained customers request and provides efficient and high-quality service to customers as per the bank’s policy, procedure, guideline and other pertinent regulation.\n\n\n=== Balanced Scorecard ===\n\nStrategic Objective: Ensure Sustainable  Profitability\nKPI: Economic Profit\ntarget: Achieve  Economic Profit Plan\n------------------------\n\nStrategic Objective: Ensure Sustainable  Profitability\nKPI: Economic Profit\ntarget: Graduating branches from loss to profit making\n------------------------\n\nStrategic Objective: Enhance  Resource Mobilization\nKPI: Incremental Deposit\ntarget: Achieve  Incremen

In [68]:
from gpt4all import GPT4All

# 'orca-mini-3b' is one of the smallest reliable models (~2GB)
# It will download automatically on the first run.
model = GPT4All("orca-mini-3b-gguf2-q4_0.gguf")

# Optional: if you have a GPU, you can specify device='gpu' or 'nvidia'
# model = GPT4All("orca-mini-3b-gguf2-q4_0.gguf", device='cpu')

with model.chat_session():
    response = model.generate("Hello GPT4All! Give me a fun fact.")
    print(response)

Downloading: 100%|██████████| 1.98G/1.98G [03:13<00:00, 10.2MiB/s]
Verifying: 100%|██████████| 1.98G/1.98G [00:03<00:00, 514MiB/s]


 Did you know that the Great Wall of China is visible from space?


In [70]:
# 1. Manager input
num_objectives = 5

# 2. Context already prepared
context = full_context

# 3. Generate prompt
prompt = generate_prompt(context, num_objectives)

# 4. Call GPT4All
response = model.generate(prompt, max_tokens=500)

print(response)

ERROR: The prompt size exceeds the context window size and cannot be processed.


In [71]:
# chatModel = ChatOpenAI(model="gpt-4o")

from langchain_community.chat_models import ChatOllama
chatModel = ChatOllama(
    model="llama3",
    temperature=0.3
)

C:\Users\user\AppData\Local\Temp\ipykernel_7352\246707874.py:4: LangChainDeprecationWarning: The class `ChatOllama` was deprecated in LangChain 0.3.1 and will be removed in 1.0.0. An updated version of the class exists in the :class:`~langchain-ollama package and should be used instead. To use it run `pip install -U :class:`~langchain-ollama` and import as `from :class:`~langchain_ollama import ChatOllama``.
  chatModel = ChatOllama(


In [ ]:
def generate_prompt(context, num_objectives):
    return f"""
You are an expert HR performance management assistant.

Based on the Job Description and Balanced Scorecard below, generate EXACTLY {num_objectives} SMART objectives.

Each objective must be:
- Specific
- Measurable
- Achievable
- Relevant
- Time-bound

Return only a numbered list.

Context:
{context}
"""

In [73]:
prompt = generate_prompt(full_context, num_objectives)

response = chatModel.invoke(prompt)

print(response.content)

Here are the 5 SMART objectives for a Branch Banking Officer:

1. **Increase Digital Banking Users**: By the end of Q2, increase the number of digital banking users by 10% compared to the previous quarter, with a target of 500 new users.

Specific: Increase digital banking users
Measurable: 10% increase in Q2, 500 new users
Achievable: Based on current trends and market conditions
Relevant: Aligns with the bank's strategy to enhance digital banking services
Time-bound: By the end of Q2

2. **Mobilize Deposits**: By the end of Q3, mobilize an additional Birr 50 million in deposits from retail customers, with a target of 20% increase in deposits compared to the previous quarter.

Specific: Mobilize deposits from retail customers
Measurable: Birr 50 million, 20% increase in Q3
Achievable: Based on current market conditions and customer demand
Relevant: Aligns with the bank's strategy to enhance resource mobilization
Time-bound: By the end of Q3

3. **Improve Customer Satisfaction**: By th